# Visualization

This notebook creates the patient-level ROC curves for reporting and run-stability boxplots from raw repeated-run outputs. The ROC curves use one mean predicted probability per test patient across 100 runs. Run-stability boxplots use the raw 100 runs for each TL model, EL configuration, and the Reference CNN.


In [ ]:
# ── User configuration ──────────────────────────────────────────────────────
DATA_DIR = 'data'                         # Directory containing raw input files
REPORTING_DIR = 'analysis_outputs'        # Directory containing evaluation tables
REPORTING_FILE = 'analysis_tables.xlsx'  # Workbook containing Patient_predictions
TL_RESULTS_FILE = 'results_TL.xlsx'       # TL workbook with 11 × 100 runs
CNN_RESULTS_FILE = 'results_CNN.xlsx'     # Reference-CNN workbook with 100 runs
EL_RESULTS_FILE = 'results_EL.xlsx'       # EL workbook with 8 × 100 runs
LABELS_FILE = 'test_labels.txt'           # Test labels in saved-score order
CLINICAL_READER_FILE = 'clinical_reader_calls.csv'  # One binary call per test patient
OUTPUT_DIR = 'figures'                    # Folder for exported figures
DPI = 300

# Change these two metric codes to create a different two-panel stability figure.
# Supported codes: 'ACC', 'PRE', 'SEN', 'SPE', 'F1S', 'AUC'.
PAPER_METRICS = ('ACC', 'F1S')
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from sklearn.metrics import accuracy_score, auc, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

mpl.rcParams.update(mpl.rcParamsDefault)
plt.style.use('default')

DATA_PATH = Path(DATA_DIR)
OUTPUT_PATH = Path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

REPORTING_PATH = Path(REPORTING_DIR) / REPORTING_FILE
TL_PATH = DATA_PATH / TL_RESULTS_FILE
CNN_PATH = DATA_PATH / CNN_RESULTS_FILE
EL_PATH = DATA_PATH / EL_RESULTS_FILE
LABELS_PATH = DATA_PATH / LABELS_FILE
READER_PATH = DATA_PATH / CLINICAL_READER_FILE

if not REPORTING_PATH.is_file():
    raise FileNotFoundError(f'Reporting workbook not found: {REPORTING_PATH}')

print('Reporting workbook:', REPORTING_PATH)
print('Output directory:', OUTPUT_PATH)


## Patient-level ROC curves

These cells create two ROC figures from patient-level probabilities. Binary-only methods are excluded because they do not provide continuous probability scores.


In [ ]:
predictions = pd.read_excel(REPORTING_PATH, sheet_name='Patient_predictions')
required_prediction_columns = {
    'method', 'family', 'within_test_order', 'true_label',
    'aggregate_score', 'score_type',
}
missing_prediction_columns = required_prediction_columns.difference(predictions.columns)
if missing_prediction_columns:
    raise ValueError(f'Patient_predictions is missing: {sorted(missing_prediction_columns)}')
predictions = predictions.sort_values(['method', 'within_test_order']).copy()
print(f'Loaded {len(predictions)} patient-level prediction rows.')


In [ ]:
TL_ORDER = [
    'DenseNet169', 'DenseNet201', 'InceptionResNetV2', 'InceptionV3', 'MobileNetV2',
    'ResNet50', 'ResNet101', 'ResNet152', 'VGG16', 'VGG19', 'Xception',
]
EL_ORDER = [
    'Top-3 Sum Rule', 'Top-3 Median Rule', 'Top-3 Max Rule',
    'Top-5 Sum Rule', 'Top-5 Median Rule', 'Top-5 Max Rule',
]
STABILITY_EL_ORDER = [
    'Top-3 Sum Rule', 'Top-3 Median Rule', 'Top-3 Max Rule', 'Top-3 Majority Voting',
    'Top-5 Sum Rule', 'Top-5 Median Rule', 'Top-5 Max Rule', 'Top-5 Majority Voting',
]
REFERENCE_CNN = 'Reference CNN'
CLINICAL_READER = 'Clinical reader'


def select_probability_methods(frame, family, order):
    selected = frame.loc[
        (frame['family'] == family) & (frame['score_type'] == 'mean probability')
    ].copy()
    available = set(selected['method'])
    missing = [method for method in order if method not in available]
    if missing:
        raise ValueError(f'Missing expected {family} methods: {missing}')
    return selected.loc[selected['method'].isin(order)].copy()


def validate_patient_labels(frame, methods):
    reference = None
    for method in methods:
        labels = (frame.loc[frame['method'].eq(method)]
                  .sort_values('within_test_order')['true_label'].to_numpy(dtype=int))
        if reference is None:
            reference = labels
        elif not np.array_equal(reference, labels):
            raise ValueError('True-label order differs across ROC methods.')
    return reference


def plot_roc_curves(frame, methods, output_name):
    line_styles = ['-', '--', '-.', ':']
    markers = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'h', 'H', '+', 'x', 'd', '|', '_']
    validate_patient_labels(frame, methods)

    default_colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', [])
    if not default_colors:
        default_colors = list(mpl.colormaps['tab10'].colors)
    method_colors = {
        method: default_colors[index % len(default_colors)]
        for index, method in enumerate(methods)
    }

    fig, ax = plt.subplots(figsize=(14, 8))
    auc_rows = []
    for index, method in enumerate(methods):
        method_data = frame.loc[frame['method'].eq(method)].sort_values('within_test_order')
        y_true = method_data['true_label'].to_numpy(dtype=int)
        y_score = method_data['aggregate_score'].to_numpy(dtype=float)
        fpr, tpr, _ = roc_curve(y_true, y_score)
        method_auc = auc(fpr, tpr)
        auc_rows.append({'Method': method, 'AUC': method_auc})

        is_reference = method == REFERENCE_CNN
        ax.plot(
            fpr, tpr,
            label=f'{method} (AUC = {method_auc:.2f})',
            color='black' if is_reference else method_colors[method],
            linestyle='-' if is_reference else line_styles[index % len(line_styles)],
            marker='o' if is_reference else markers[index % len(markers)],
            markersize=4 if is_reference else None,
            markevery=1 if is_reference else (index % 10, 10),
            linewidth=2.0 if is_reference else 1.5,
            solid_capstyle='round',
            zorder=5 if is_reference else 2,
        )

    ax.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--', label='Random classification')
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.05)
    ax.set_xlabel('False Positive Rate', fontsize=20, fontweight='bold', labelpad=15)
    ax.set_ylabel('True Positive Rate', fontsize=20, fontweight='bold', labelpad=15)
    ax.legend(title='Models', bbox_to_anchor=(1.05, 0.5), loc='center left', fontsize=18, title_fontsize=18)
    ax.tick_params(axis='both', labelsize=20)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight('bold')
    ax.set_yticks(np.arange(0.2, 1.01, 0.2))
    ax.yaxis.set_minor_locator(mticker.NullLocator())
    ax.grid(True, linewidth=0.8, alpha=0.7)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
    fig.tight_layout(pad=3.0)

    png_path = OUTPUT_PATH / f'{output_name}.png'
    pdf_path = OUTPUT_PATH / f'{output_name}.pdf'
    fig.savefig(png_path, dpi=DPI, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    plt.show()
    display(pd.DataFrame(auc_rows).style.format({'AUC': '{:.3f}'}))
    print(f'Saved: {png_path}')
    print(f'Saved: {pdf_path}')


In [ ]:
tl_predictions = select_probability_methods(predictions, 'TL', TL_ORDER)
cnn_predictions = predictions.loc[
    (predictions['family'] == 'CNN') &
    (predictions['method'] == REFERENCE_CNN) &
    (predictions['score_type'] == 'mean probability')
].copy()
if cnn_predictions.empty:
    raise ValueError(f'{REFERENCE_CNN} mean probabilities were not found.')

plot_roc_curves(
    pd.concat([tl_predictions, cnn_predictions], ignore_index=True),
    TL_ORDER + [REFERENCE_CNN],
    'ROC_TL_reference_CNN_patient_level',
)


In [ ]:
el_predictions = select_probability_methods(predictions, 'EL', EL_ORDER)
plot_roc_curves(
    pd.concat([el_predictions, cnn_predictions], ignore_index=True),
    EL_ORDER + [REFERENCE_CNN],
    'ROC_EL_reference_CNN_patient_level',
)


## Run stability across 100 runs

The following cells read the three raw repeated-run workbooks. They create (1) a configurable two-metric figure for the manuscript and (2) six separate figures, one for each metric, for complete visual inspection.


In [ ]:
METRIC_LABELS = {
    'ACC': 'ACC',
    'PRE': 'PRE',
    'SEN': 'SEN',
    'SPE': 'SPE',
    'F1S': 'F1S',
    'AUC': 'AUC',
}
ALL_METRICS = tuple(METRIC_LABELS)


def read_run_workbook(path):
    sheets = pd.read_excel(path, sheet_name=None, engine='openpyxl')
    run_sheet = 'Run_results' if 'Run_results' in sheets else next(iter(sheets))
    return sheets, sheets[run_sheet].copy()


def read_test_labels(path):
    values = path.read_text(encoding='utf-8').strip().replace(',', ' ').split()
    if not values:
        raise ValueError(f'{path.name} does not contain test labels.')
    labels = np.asarray([int(value) for value in values], dtype=int)
    if set(np.unique(labels)) != {0, 1}:
        raise ValueError(f'{path.name} must contain both binary outcome classes.')
    return labels


def require_columns(frame, columns, source_name):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise ValueError(f'{source_name} is missing columns: {missing}')


def first_available_column(frame, candidates, source_name):
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate
    raise ValueError(f'{source_name} is missing all supported alternatives: {candidates}')


def parse_vector(value, expected_length, field_name):
    text = str(value).strip().strip('[](){} ')
    if not text:
        raise ValueError(f'{field_name} is empty.')
    parts = text.split(',') if ',' in text else text.split()
    values = np.asarray([float(part) for part in parts if part.strip()], dtype=float)
    if len(values) != expected_length:
        raise ValueError(f'{field_name} contains {len(values)} values; expected {expected_length}.')
    return values


def test_manifest_labels(sheets, source_name):
    if 'Patient_manifest' not in sheets:
        return None
    manifest = sheets['Patient_manifest'].copy()
    require_columns(manifest, ['split', 'within_split_order', 'observed_label'], source_name + ' Patient_manifest')
    test = manifest.loc[manifest['split'].astype(str).str.lower().eq('test')].copy()
    test = test.sort_values('within_split_order', kind='stable').reset_index(drop=True)
    if test.empty:
        raise ValueError(f'{source_name} Patient_manifest has no test rows.')
    return pd.to_numeric(test['observed_label'], errors='raise').to_numpy(dtype=int)


def auc_from_vectors(frame, n_patients, source_name):
    probability_column = first_available_column(
        frame, ['test_probabilities', 'test_predicts', 'predicts'], source_name
    )
    return [
        roc_auc_score(y_true, parse_vector(value, n_patients, f'{source_name} probability vector'))
        for value in frame[probability_column]
    ]


def extract_metric(frame, candidates, source_name):
    column = first_available_column(frame, candidates, source_name)
    return pd.to_numeric(frame[column], errors='raise')


def normalise_tl_runs(frame, n_patients):
    require_columns(frame, ['model_name', 'test_accuracy'], 'TL results workbook')
    run_column = first_available_column(frame, ['seed', 'tag'], 'TL results workbook')
    auc_values = frame['test_auc'] if 'test_auc' in frame.columns else auc_from_vectors(frame, n_patients, 'TL results workbook')
    output = pd.DataFrame({
        'method': frame['model_name'].astype(str),
        'group': 'TL',
        'run_id': frame[run_column],
        'ACC': extract_metric(frame, ['test_accuracy'], 'TL results workbook'),
        'PRE': extract_metric(frame, ['test_precision'], 'TL results workbook'),
        'SEN': extract_metric(frame, ['test_sensitivity', 'test_recall'], 'TL results workbook'),
        'SPE': extract_metric(frame, ['test_specificity'], 'TL results workbook'),
        'F1S': extract_metric(frame, ['test_f1', 'test_f1score'], 'TL results workbook'),
        'AUC': pd.to_numeric(auc_values, errors='raise'),
    })
    if run_column == 'tag' and output.duplicated(['method', 'run_id']).any():
        output['run_id'] = output.groupby('method', sort=False).cumcount() + 1
    return output


def normalise_cnn_runs(frame):
    require_columns(frame, ['seed', 'test_accuracy', 'test_precision', 'test_sensitivity', 'test_specificity', 'test_f1', 'test_auc'], 'CNN Run_results')
    return pd.DataFrame({
        'method': REFERENCE_CNN,
        'group': 'Reference CNN',
        'run_id': frame['seed'],
        'ACC': pd.to_numeric(frame['test_accuracy'], errors='raise'),
        'PRE': pd.to_numeric(frame['test_precision'], errors='raise'),
        'SEN': pd.to_numeric(frame['test_sensitivity'], errors='raise'),
        'SPE': pd.to_numeric(frame['test_specificity'], errors='raise'),
        'F1S': pd.to_numeric(frame['test_f1'], errors='raise'),
        'AUC': pd.to_numeric(frame['test_auc'], errors='raise'),
    })


def normalise_rule_name(value):
    labels = {
        'sum': 'Sum Rule', 'sum rule': 'Sum Rule',
        'median': 'Median Rule', 'median rule': 'Median Rule',
        'max': 'Max Rule', 'max rule': 'Max Rule',
        'majority voting': 'Majority Voting', 'majority_vote': 'Majority Voting',
        'majority vote': 'Majority Voting',
    }
    key = str(value).strip().lower()
    if key not in labels:
        raise ValueError(f'Unsupported ensemble rule: {value}')
    return labels[key]


def normalise_el_runs(frame):
    require_columns(frame, ['pool', 'rule', 'tag', 'test_accuracy', 'test_precision', 'test_sensitivity', 'test_specificity'], 'EL Run_results')
    rules = frame['rule'].map(normalise_rule_name)
    auc_values = frame['test_auc'] if 'test_auc' in frame.columns else np.nan
    return pd.DataFrame({
        'method': frame['pool'].astype(str) + ' ' + rules,
        'group': 'EL',
        'run_id': frame['tag'],
        'ACC': pd.to_numeric(frame['test_accuracy'], errors='raise'),
        'PRE': pd.to_numeric(frame['test_precision'], errors='raise'),
        'SEN': pd.to_numeric(frame['test_sensitivity'], errors='raise'),
        'SPE': pd.to_numeric(frame['test_specificity'], errors='raise'),
        'F1S': extract_metric(frame, ['test_f1', 'test_f1score'], 'EL Run_results'),
        'AUC': pd.to_numeric(auc_values, errors='coerce'),
    })


def validate_run_coverage(frame, expected_runs=100):
    counts = frame.groupby('method', sort=True).size()
    invalid = counts[counts.ne(expected_runs)]
    if not invalid.empty:
        raise ValueError(f'Each method must contain {expected_runs} runs. Invalid coverage:\n{invalid}')
    duplicates = frame.duplicated(['method', 'run_id'], keep=False)
    if duplicates.any():
        raise ValueError('Duplicate method/run identifiers were found.')


def validate_metrics(frame):
    for metric in ['ACC', 'PRE', 'SEN', 'SPE', 'F1S']:
        values = frame[metric]
        if values.isna().any() or ((values < 0) | (values > 1)).any():
            raise ValueError(f'{metric} contains missing or out-of-range values.')
    auc_values = frame.loc[~frame['method'].str.contains('Majority Voting', na=False), 'AUC']
    if auc_values.isna().any() or ((auc_values < 0) | (auc_values > 1)).any():
        raise ValueError('AUC contains missing or out-of-range values for probability-producing methods.')


In [ ]:
stability_paths = [TL_PATH, CNN_PATH, EL_PATH, LABELS_PATH, READER_PATH]
missing_stability_paths = [path for path in stability_paths if not path.is_file()]
if missing_stability_paths:
    raise FileNotFoundError('Missing stability inputs:\n' + '\n'.join(str(path) for path in missing_stability_paths))

tl_sheets, tl_raw = read_run_workbook(TL_PATH)
cnn_sheets, cnn_raw = read_run_workbook(CNN_PATH)
el_sheets, el_raw = read_run_workbook(EL_PATH)
y_true = read_test_labels(LABELS_PATH)

for source_name, sheets in [('TL', tl_sheets), ('Reference CNN', cnn_sheets)]:
    manifest_labels = test_manifest_labels(sheets, source_name)
    if manifest_labels is not None and not np.array_equal(manifest_labels, y_true):
        raise ValueError(f'{source_name} test-patient order does not match LABELS_FILE.')

runs = pd.concat(
    [normalise_tl_runs(tl_raw, len(y_true)), normalise_el_runs(el_raw), normalise_cnn_runs(cnn_raw)],
    ignore_index=True,
)
runs['run_id'] = pd.to_numeric(runs['run_id'], errors='raise').astype(int)
validate_run_coverage(runs)
validate_metrics(runs)

reader = pd.read_csv(READER_PATH).sort_values('within_test_order', kind='stable').reset_index(drop=True)
require_columns(reader, ['within_test_order', 'clinical_reader_call'], 'Clinical reader file')
if len(reader) != len(y_true) or not np.array_equal(reader['within_test_order'].to_numpy(), np.arange(1, len(y_true) + 1)):
    raise ValueError('Clinical reader file does not contain one ordered call per test patient.')
reader_calls = pd.to_numeric(reader['clinical_reader_call'], errors='raise').to_numpy(dtype=int)
if not set(np.unique(reader_calls)).issubset({0, 1}):
    raise ValueError('Clinical reader calls must be binary.')
tn, fp, fn, tp = confusion_matrix(y_true, reader_calls, labels=[0, 1]).ravel()
clinical_metrics = {
    'ACC': accuracy_score(y_true, reader_calls),
    'PRE': precision_score(y_true, reader_calls, zero_division=0),
    'SEN': recall_score(y_true, reader_calls, zero_division=0),
    'SPE': tn / (tn + fp),
    'F1S': f1_score(y_true, reader_calls, zero_division=0),
    'AUC': np.nan,
}

STABILITY_METHOD_ORDER = TL_ORDER + STABILITY_EL_ORDER + [REFERENCE_CNN, CLINICAL_READER]
missing_methods = [method for method in STABILITY_METHOD_ORDER[:-1] if method not in set(runs['method'])]
if missing_methods:
    raise ValueError(f'Missing expected repeated-run methods: {missing_methods}')

METHOD_GROUPS = {
    **{method: 'TL' for method in TL_ORDER},
    **{method: 'EL' for method in STABILITY_EL_ORDER},
    REFERENCE_CNN: 'Reference CNN',
    CLINICAL_READER: 'Clinical reader',
}
GROUP_STYLE = {
    'TL': {'facecolor': '#FCE4C4', 'edgecolor': '#E5A146'},
    'EL': {'facecolor': '#CDD7F8', 'edgecolor': '#3347D3'},
    'Reference CNN': {'facecolor': '#D9D9D9', 'edgecolor': '#777777'},
}

print(f'Validated {len(runs)} raw repeated runs across {runs["method"].nunique()} methods.')
print(runs.groupby('group')['method'].nunique().to_string())
print('Clinical reader:', {metric: f'{value:.2f}' for metric, value in clinical_metrics.items() if pd.notna(value)})


In [ ]:
DISPLAY_TICKS = np.arange(0.0, 1.01, 0.2)


def summary_table(frame):
    rows = []

    for method in STABILITY_METHOD_ORDER[:-1]:
        row = {'Method': method}
        method_data = frame.loc[frame['method'].eq(method)]

        for metric in ALL_METRICS:
            values = method_data[metric].dropna().to_numpy(dtype=float)
            row[f'{metric} median'] = np.median(values) if len(values) else np.nan
            row[f'{metric} IQR'] = (
                np.percentile(values, 75) - np.percentile(values, 25)
                if len(values) else np.nan
            )

        rows.append(row)

    rows.append({
        'Method': CLINICAL_READER,
        **{f'{metric} median': clinical_metrics[metric] for metric in ALL_METRICS},
        **{f'{metric} IQR': np.nan for metric in ALL_METRICS},
    })

    return pd.DataFrame(rows)


def methods_for_metric(frame, metric):
    methods = []

    for method in STABILITY_METHOD_ORDER:
        if method == CLINICAL_READER:
            if pd.notna(clinical_metrics[metric]):
                methods.append(method)
        elif frame.loc[frame['method'].eq(method), metric].notna().any():
            methods.append(method)

    return methods


def add_group_lines(ax, methods):
    for index in range(len(methods) - 1):
        if METHOD_GROUPS[methods[index]] != METHOD_GROUPS[methods[index + 1]]:
            y_position = len(methods) - index - 0.5
            ax.axhline(y_position, color='#B8B8B8', linewidth=0.9)


def draw_stability_panel(ax, frame, metric, xlim=(0.0, 1.0), show_y_labels=True):
    methods = methods_for_metric(frame, metric)
    positions = np.arange(len(methods), 0, -1)

    for position, method in zip(positions, methods):
        if method == CLINICAL_READER:
            ax.plot(
                clinical_metrics[metric],
                position,
                marker='o',
                color='red',
                markersize=8,
                markeredgecolor='white',
                markeredgewidth=0.8,
                zorder=6,
            )
            median_text = f'{clinical_metrics[metric]:.2f}'

        else:
            values = frame.loc[frame['method'].eq(method), metric].dropna().to_numpy(dtype=float)
            style = GROUP_STYLE[METHOD_GROUPS[method]]

            ax.boxplot(
                values,
                positions=[position],
                vert=False,
                widths=0.55,
                patch_artist=True,
                manage_ticks=False,
                whis=1.5,
                showfliers=True,
                medianprops={
                    'color': style['edgecolor'],
                    'linewidth': 1.8,
                },
                boxprops={
                    'facecolor': style['facecolor'],
                    'edgecolor': style['edgecolor'],
                    'linewidth': 1.2,
                },
                whiskerprops={
                    'color': '#666666',
                    'linewidth': 1.0,
                },
                capprops={
                    'color': '#666666',
                    'linewidth': 1.0,
                },
                flierprops={
                    'marker': '+',
                    'markeredgecolor': '#8A8A8A',
                    'markersize': 7,
                    'alpha': 0.85,
                },
            )
            median_text = f'{np.median(values):.2f}'

        ax.text(
            1.035,
            position,
            median_text,
            transform=ax.get_yaxis_transform(),
            va='center',
            ha='left',
            fontsize=14,
            clip_on=False,
        )

    add_group_lines(ax, methods)

    visible_ticks = [tick for tick in DISPLAY_TICKS if xlim[0] <= tick <= xlim[1]]
    ax.set_xlim(*xlim)
    ax.set_xticks(visible_ticks)
    ax.set_xlabel(
        f'{METRIC_LABELS[metric]} across 100 runs',
        fontsize=20,
        fontweight='bold',
        labelpad=15,
    )
    ax.tick_params(axis='x', labelsize=15, width=0.8)
    ax.grid(axis='x', linewidth=0.8, alpha=0.7)
    ax.grid(axis='y', visible=False)

    ax.set_yticks(positions)
    if show_y_labels:
        ax.set_yticklabels(methods, fontsize=14, fontweight='bold')
    else:
        ax.tick_params(axis='y', left=False, labelleft=False)

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

    ax.text(
        1.035,
        len(methods) + 0.75,
        'median',
        transform=ax.get_yaxis_transform(),
        va='bottom',
        ha='left',
        fontsize=14,
        fontstyle='italic',
        clip_on=False,
    )


def stability_legend(fig):
    handles = [
        Patch(
            facecolor=GROUP_STYLE['TL']['facecolor'],
            edgecolor=GROUP_STYLE['TL']['edgecolor'],
            label='TL',
        ),
        Patch(
            facecolor=GROUP_STYLE['EL']['facecolor'],
            edgecolor=GROUP_STYLE['EL']['edgecolor'],
            label='EL',
        ),
        Patch(
            facecolor=GROUP_STYLE['Reference CNN']['facecolor'],
            edgecolor=GROUP_STYLE['Reference CNN']['edgecolor'],
            label='Reference CNN',
        ),
        Line2D(
            [0],
            [0],
            marker='o',
            color='red',
            markerfacecolor='red',
            markeredgecolor='white',
            markersize=8,
            linewidth=0,
            label='Clinical reader (single reading)',
        ),
        Line2D(
            [0],
            [0],
            marker='+',
            color='#8A8A8A',
            markersize=8,
            linewidth=0,
            label='Outliers',
        ),
    ]

    fig.legend(
        handles=handles,
        title='Methods',
        loc='lower center',
        ncol=5,
        fontsize=14,
        title_fontsize=14,
        frameon=True,
        bbox_to_anchor=(0.5, 0.008),
        borderaxespad=0,
    )


def save_figure(fig, output_name):
    png_path = OUTPUT_PATH / f'{output_name}.png'
    pdf_path = OUTPUT_PATH / f'{output_name}.pdf'

    fig.savefig(png_path, dpi=DPI, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    plt.show()

    print(f'Saved: {png_path}')
    print(f'Saved: {pdf_path}')


def plot_metric_pair(frame, left_metric, right_metric, output_name=None, xlim=(0.0, 1.0)):
    for metric in [left_metric, right_metric]:
        if metric not in METRIC_LABELS:
            raise ValueError(
                f'Unsupported metric: {metric}. Choose from {tuple(METRIC_LABELS)}.'
            )

    left_methods = methods_for_metric(frame, left_metric)
    right_methods = methods_for_metric(frame, right_metric)
    shared_methods = left_methods == right_methods

    fig, axes = plt.subplots(1, 2, figsize=(16, 12), sharey=shared_methods)

    draw_stability_panel(
        axes[0],
        frame,
        left_metric,
        xlim=xlim,
        show_y_labels=True,
    )
    draw_stability_panel(
        axes[1],
        frame,
        right_metric,
        xlim=xlim,
        show_y_labels=not shared_methods,
    )

    stability_legend(fig)
    fig.subplots_adjust(
        left=0.25,
        right=0.90,
        bottom=0.23,
        top=0.95,
        wspace=0.20,
    )

    name = output_name or f'Run_stability_{left_metric}_{right_metric}'
    save_figure(fig, name)


def plot_single_metric(frame, metric, output_name=None, xlim=(0.0, 1.0)):
    if metric not in METRIC_LABELS:
        raise ValueError(
            f'Unsupported metric: {metric}. Choose from {tuple(METRIC_LABELS)}.'
        )

    fig, ax = plt.subplots(figsize=(10, 12))

    draw_stability_panel(
        ax,
        frame,
        metric,
        xlim=xlim,
        show_y_labels=True,
    )

    stability_legend(fig)
    fig.subplots_adjust(
        left=0.36,
        right=0.87,
        bottom=0.18,
        top=0.95,
    )

    save_figure(fig, output_name or f'Run_stability_{metric}')


In [ ]:
stability_summary = summary_table(runs)

display(
    stability_summary.style.format({
        column: '{:.2f}'
        for column in stability_summary.columns
        if column.endswith(('median', 'IQR'))
    })
)

PAPER_XLIM = (0.2, 1.0)

plot_metric_pair(
    runs,
    PAPER_METRICS[0],
    PAPER_METRICS[1],
    output_name=f'Run_stability_{PAPER_METRICS[0]}_{PAPER_METRICS[1]}',
    xlim=PAPER_XLIM,
)


In [ ]:
for metric in ALL_METRICS:
    plot_single_metric(
        runs,
        metric,
        output_name=f'Run_stability_{metric}',
        xlim=(0.0, 1.0),
    )


The paired figure is intended for the manuscript. Change `PAPER_METRICS` in the configuration cell to generate any supported two-metric combination. The final cell creates six separate figures for visual inspection; they need not be included in the manuscript or supplementary material.
